# 00: Data from Binance

Fetches the three raw datasets from Binance's public API (cached under
`data/raw/`) and writes the processed panels that notebooks 01-06 read.

Windows used throughout: development 2020-01 to 2024-07, gate 2024-08 to
2025-06, lockbox 2025-07 to the end of the data. Only notebook 06 reads the
lockbox.

In [1]:
from pathlib import Path

In [2]:
import numpy as np
import pandas as pd

In [3]:
import sys
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))
from quantlib import data, fetch

In [4]:
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW, PROC = ROOT / "data" / "raw", ROOT / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)

In [5]:
FORCE = False  # set True to refetch regardless of cache age

## Daily panel (top 150 pairs, 2018 to today)

In [6]:
raw_1d = fetch.refresh(
    RAW / "binance_1d.pkl.zip",
    lambda: fetch.fetch_binance(top_n=150, start="2018-01-01"),
    force=FORCE,
)
panels = data.to_panels(raw_1d)
print(raw_1d.shape, raw_1d.index.min().date(), "->", raw_1d.index.max().date())

(3109, 447) 2018-01-01 -> 2026-07-06


## Point-in-time universe: top 100 by trailing 30-day median ADV, $1M floor
The volume is Binance-only, so the floor is lower than an all-exchange
figure would need to be.

In [7]:
universe = data.build_universe(panels, top_n=100, min_adv_usd=1_000_000)
print("median universe size:", int(universe.loc["2020-01-01":].sum(axis=1).median()))

median universe size: 66


## Hourly panel (top 60 by current ADV, 2020 to today)
Only the names the intraday work could realistically trade.

In [8]:
adv_now = panels["dollar_volume"].rolling(30).median().iloc[-1]
top60 = adv_now.dropna().sort_values(ascending=False).head(60).index.tolist()

In [9]:
raw_1h = fetch.refresh(
    RAW / "binance_1h.pkl.zip",
    lambda: fetch.fetch_binance(symbols=top60, start="2020-01-01", interval="1h"),
    force=FORCE,
)
panels_1h = data.to_panels(raw_1h)
print(raw_1h.shape)

(57041, 180)


## Funding rates (the same 60 names, where a perp exists; 2019-09 to today)

In [10]:
funding = fetch.refresh(
    RAW / "funding.pkl.zip",
    lambda: fetch.fetch_funding(top60, start="2019-09-01"),
    force=FORCE,
)
print("funding coverage:", funding.shape)

funding coverage: (2492, 55)


## Benchmarks: BTC and the equal-weight tradable market

In [12]:
benchmarks = pd.DataFrame({
    "BTC": panels["returns"]["BTC"],
    "MKT": data.market_return(panels, universe),
})

## Save the processed panels

In [13]:
panels["price"].to_parquet(PROC / "price.parquet")
panels["returns"].to_parquet(PROC / "returns.parquet")
panels["dollar_volume"].to_parquet(PROC / "dollar_volume.parquet")
panels["taker_imbalance"].to_parquet(PROC / "taker_imbalance.parquet")
universe.to_parquet(PROC / "universe.parquet")
benchmarks.to_parquet(PROC / "benchmarks.parquet")
funding.to_parquet(PROC / "funding.parquet")
panels_1h["price"].to_parquet(PROC / "price_1h.parquet")
panels_1h["dollar_volume"].to_parquet(PROC / "volume_1h.parquet")
panels_1h["taker_imbalance"].to_parquet(PROC / "taker_1h.parquet")
print("saved", len(list(PROC.glob("*.parquet"))), "parquets")

saved 15 parquets
